# Фичи, кластеризация, эмбеддинги

## Годовой проект. Чекпойнт 5

Пятый чекпойнт посвящен использованию более сложных моделей и архитектур в задаче вашего годового проекта.

## Трек ML:

От вас ожидается улучшение решения путем:
* Использования нелинейных моделей ML (деревья, леса и бустинги)
* Работа с признаками - придумать новые признаки, удалить лишние, использовать кластеризацию для получения дополнительных признаков и так далее (feature-engineering)

В результате экспериментов с моделями и признаками необходимо составить таблицу с результатами экспериментов (отобразить модели с гиперпараметрами, результаты метрик, а также сравнить время обучения моделей). 

Опишите лучшее решение и попробуйте сделать выводы, почему решение оказалось лучшим.


In [ ]:
import io
import warnings
import numpy as np
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 
from src.services.utils import Boto3Reader

import pandas as pd

from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import MinMaxScaler, StandardScaler

from xgboost import XGBClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

import json
from scipy.stats import chi2_contingency
from pathlib import Path 


from tqdm import tqdm
import gc
from sklearn.decomposition import TruncatedSVD


warnings.simplefilter(action='ignore', category=FutureWarning)

## Содержание 

## Загрузка датасетов 

In [ ]:
# USE_MLFLOW = True 
USE_S3 = False

```USE_S3=True``` => использование `config_s3.json`, иначе чтение .csv локально

In [ ]:
from src.services.utils import Boto3Reader

# local 
local_filename = 'dump_features_include_numeric_1.csv'
config_path = 'src/config_s3.json'

if USE_S3: 
    reader = Boto3Reader(config_path)
    df = reader.get_boto3_csv(
        object_path='dump_features_include_numeric_1.csv'
    )

    # clear memory and connection: 
    del reader
else: 
    df = pd.read_csv(local_filename, index_col=0)

In [ ]:
df.head(3)

## 1. Отбор признаков 
### 1.1 Отбор лингвистических признаков по порогу корреляции

In [ ]:
# Calc composed features correlations woth target: 
composed_features = [
    'count_spp',
    'count_rpp',
    'punct_after_space',
    'has_emoji',
    'has_emoticon',
    'has_capslock',
    'is_all_lower',
    'has_punctuation_spp',
    'has_punctuation_rpp',
    'has_fence_ironic_style',
    'count_profanity',
    'has_pronouns',
    'starts_with_cap',
    'has_url',
    'has_number',
    'has_mention',
    'has_hashtag',
    # 'has_email',
    'ends_with_dot',
    'has_emotional_sym',
    # 'has_repeating_chars',
    'has_repeating_letters_3plus'
]

df_corr = df[df['is_toxic']!=np.nan]
correlations = df_corr[composed_features + ['is_toxic']].corr()['is_toxic'].drop('is_toxic')

# Plot tops of the correlated: 
correlations_sorted = correlations.sort_values(ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x=correlations_sorted.values, y=correlations_sorted.index, palette="viridis")
plt.title("Корреляция лингвистических фичей \nс таргетом", fontsize=18)
plt.xlabel('Значение', fontsize=16)
plt.ylabel('Тип фичи', fontsize=16)
plt.xlim(-0.3, 0.3)  
plt.tight_layout()
plt.grid(alpha=0.5)
plt.show()

In [ ]:
# use features with corr >=0.5 only: 
corr_bound = 0.05
corr_abs: pd.Series = correlations.abs().sort_values(ascending=False)
high_corr_features = corr_abs[corr_abs>=corr_bound].index
print(f'Признаки с корреляцией >= {corr_bound} к таргету:\n {high_corr_features.values}')

**Вывод**   
* Корреляция с таргетом от `0.05` до `0.3` это много, фичи стоит использовать
* С корреляцией < 0.05 фичи можно выкинуть для линейных моделей, для нелинейных стоит оставить

### 1.2 Отбор категориальных признаков по χ^2
Как влияют отдельные эмотиконы / эмодзи / последовательности символов на таргет?  
ONE_HOT энкодинг -> Сохранили в файлы. Посчитаем по ним χ^2.

In [ ]:


# one-hot encoding files by all datasets rows: 
ohe_files = [
    # 'df_profanities_new.csv', # danger zone 
    'df_emoticon_new.csv',
    'df_emoji_new.csv',
    # 'df_rep_punct_new.csv'
]
n_tokens = 30

# USE_S3 = True # for testing local files
for file in ohe_files:
    # file key: 
    file_key = Path(file).stem.replace('df_', '').replace('_new', '')
    enc_name = 'encoding_' + file_key + '.json'

    # read data: 
    if USE_S3:
        reader = Boto3Reader(config_path, bucket_name='toxic-messages-bucket-1')
        df_ohe = reader.get_boto3_csv(
            object_path='OHE_by_tokens/'+file
            )
        enc_dict = reader.get_boto3_json(
            object_path='OHE_by_tokens/'+enc_name
        )
    else: 
        df_ohe = pd.read_csv(file, index_col=0, encoding='utf-8')
        with open(enc_name, 'r', encoding='utf-8') as f: 
            enc_dict = json.load(f) 

    # get most frequent tokens:
    tokens = df_ohe.columns
    token_counts = df_ohe.sum().sort_values(ascending=False)[:n_tokens]

    # calculate khi^2 of OHE features with target, for n_tokens features:
    chi2_contingency_values = []
    for token in token_counts.index:
        contingency_table = pd.crosstab(df_ohe[token], df['is_toxic'])
        chi2, p, dof, expected = chi2_contingency(contingency_table)
        chi2_contingency_values.append((token, chi2))

    # sort by chi^2 and plot:
    chi2_contingency_values.sort(key=lambda x: x[1], reverse=True)
    tokens_sorted, chi2_values_sorted = zip(*chi2_contingency_values)

    init_symbs = list()
    # decode tokens into symbols for nice plots: 
    for token in tokens_sorted:
        init_symbs.append(
            [init_symb for init_symb in enc_dict.keys() if enc_dict.get(init_symb)==token][0]
        )
    tokens_sorted = init_symbs

    plt.figure(figsize=(8,5))
    sns.barplot(x=chi2_values_sorted, y=tokens_sorted, palette="viridis")

    plt.title(f"Chi^2 по {file_key} и таргету", fontsize=18)
    plt.xlabel('Значение', fontsize=16)
    plt.ylabel('Токен', fontsize=16)
    plt.xlim(0, max(chi2_values_sorted)*1.1)
    plt.tight_layout()
    plt.grid(alpha=0.5)

    del df_ohe

**Выводы**
* Вывели, какие эмотиконы/эмодзи сильнее всего связаны с таргетом
* Удалять ничего не будем, т.к. regexp почти не влияяют на скорость препроцессинга

## 2. Визуализация кластеров

In [ ]:
preproc_cols = ['text_raw', 'text_encoded_profanity', 'text_del_stop_words', 'text_without_tokens', 'text_save_inform']

for col in preproc_cols: 
    df.dropna(subset=[col], inplace=True)
df.dropna(subset=['is_toxic'], inplace=True)

print(df.shape)

### 2.1 SVD

Используем SVD для визуализации, чтобы ответить на вопросы: 
* Как влияет тип препроцессинга на разделимость классов?
* Какое кол-во фичей в TF-IDF лучшее для разделимости классов? 
* Влияет ли добавление лингвистических фичей на разделимость классов?

In [ ]:


td_idf_n_features = [1000, 5000, 10000]
fig, axes = plt.subplots(figsize=(18, 14), nrows=3, ncols=len(preproc_cols))
plt.suptitle('SVD\nдля разных видов препроцессинга текста\nбез кастомных фичей', fontsize=20, y=1.02)

for i_col, train_column in tqdm(enumerate(preproc_cols), desc="Processing text columns"):
    
    for i_row, n_vec_features in tqdm(enumerate(td_idf_n_features), 
                                      desc=f"Processing features for {train_column}"):
        vectorizer = TfidfVectorizer(max_features=n_vec_features)
        X = vectorizer.fit_transform(df[train_column])

        svd = TruncatedSVD(n_components=2, random_state=42)
        X_svd = svd.fit_transform(X)

        ax = axes[i_row, i_col]

        sns.scatterplot(
            x=X_svd[:, 0], 
            y=X_svd[:, 1], 
            hue=df['is_toxic'], 
            palette='viridis', 
            alpha=0.7,
            ax=ax)

        # ax.set_title(f"{n_vec_features} признаков TF-IDF", fontsize=10)

        if i_row == 2:
            ax.set_xlabel(f'{train_column}\nSVD Component 1', fontsize=16)
        if i_col == 0:
            ax.set_ylabel(f'{n_vec_features} фичей\nSVD Component 2', fontsize=16)

        ax.legend(title='is_toxic', loc='best')
        gc.collect()

plt.tight_layout()
plt.show()

In [ ]:


num_features = df[composed_features]
# num_features_norm = StandardScaler().fit_transform(num_features)

td_idf_n_features = [1000, 5000, 10000]
fig, axes = plt.subplots(figsize=(18, 14), nrows=3, ncols=len(preproc_cols))
plt.suptitle('SVD\nдля разных видов препроцессинга текста\nс нормализованными кастомными фичами', fontsize=20, y=1.02)

for i_col, train_column in tqdm(enumerate(preproc_cols), desc="Processing text columns"):
    
    for i_row, n_vec_features in tqdm(enumerate(td_idf_n_features), 
                                      desc=f"Processing features for {train_column}"):
        vectorizer = TfidfVectorizer(max_features=n_vec_features)
        X = vectorizer.fit_transform(df[train_column])

        # tf-idf is sparse, so compose numeric features into sparse format too: 
        num_features_norm = MinMaxScaler(
            feature_range=(X.min(), X.max())
        ).fit_transform(num_features)
        num_features_norm = num_features_norm * 0.1 
        num_sparse = csr_matrix(num_features_norm)
        num_shape = num_sparse.shape

        # print(X.max())
        # concat vectorized text and linguistic features: 
        X = hstack([X, num_sparse])

        svd = TruncatedSVD(n_components=2, random_state=42)
        X_svd = svd.fit_transform(X)

        ax = axes[i_row, i_col]

        sns.scatterplot(
            x=X_svd[:, 0], 
            y=X_svd[:, 1], 
            hue=df['is_toxic'], 
            palette='viridis', 
            alpha=0.7,
            ax=ax)

        # ax.set_title(f"{n_vec_features} признаков TF-IDF", fontsize=10)

        if i_row == 2:
            ax.set_xlabel(f'{train_column}\nSVD Component 1', fontsize=16)
        if i_col == 0:
            ax.set_ylabel(f'{n_vec_features} фичей\nSVD Component 2', fontsize=16)

        ax.legend(title='is_toxic', loc='best')
        gc.collect()
        # plt.show()

plt.tight_layout()
plt.show()

**Выводы**  

Где кластеры самые плотные: 
* Колонка `text_without_tokens` 
* X = `TF-IDF(n_features=10000)`

### 2.2 Кластеризация K-Means

## 3. Нелинейные модели ML

### 3.1 XGBoost

In [ ]:
from sklearn import pipeline
from sklearn.preprocessing import MinMaxScaler
# from sklearn.pipeline import FunctionTransformer


train_col = 'text_without_tokens'
num_features = df[composed_features]

# add class weigths:
class_weights = df['is_toxic'].value_counts(normalize=True).to_dict()
class_weights = {0: class_weights.get(0, 0), 1: class_weights.get(1, 0)} 

vectorizer = TfidfVectorizer(max_features=10000)
X = vectorizer.fit_transform(df[train_col])
num_features_norm = MinMaxScaler(feature_range=(X.min(), X.max())).fit_transform(num_features)
num_features_norm = num_features_norm * 0.1 
X = csr_matrix(num_features_norm)

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    df['is_toxic'], 
    test_size=0.33, 
    random_state=42
    )


In [ ]:

xgb = XGBClassifier(scale_pos_weight=class_weights[0] / class_weights[1], random_state=42)
xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)

print(classification_report(y_test, y_pred, zero_division=0))

**Выводы**
* Перебирать по топ метрикам было лучше